# Multimodal LLMs

Deep Dive 08. The [main LLM API notebook](/courses/llm-eng/01-llm-api.html) focuses entirely on text. In practice, financial document workflows are overwhelmingly visual: quarterly earnings decks are PowerPoint slides rendered as images, annual reports are PDFs whose tables and charts resist clean text extraction, and scanned filings introduce noise that breaks traditional parsers. This deep dive covers the vision capability of GPT-4o and shows how to extend the RAG pipeline from [notebook 07](/courses/llm-eng/07-rag-pipeline.html) to handle mixed text and image content — ending with an end-to-end earnings analyst assistant that processes a synthetic three-page earnings presentation, indexes it in a multimodal retrieval store, and answers analyst questions that require reading both charts and prose.

Setup:

In [ ]:
#| echo: false
import os, json, base64, io, time, textwrap
import numpy as np
from dotenv import load_dotenv
load_dotenv()

import openai
from pydantic import BaseModel, Field
from typing import Optional, Type, Literal

PRICES = {
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
}

class LLMClient:
    def __init__(self, model="gpt-4o-mini", temperature=0.0):
        self.model = model; self.temperature = temperature
        self._client = openai.OpenAI()
        self._in = 0; self._out = 0

    def complete(self, messages, *, response_format=None):
        if response_format is not None:
            resp = self._client.beta.chat.completions.parse(
                model=self.model, messages=messages,
                temperature=self.temperature, response_format=response_format)
        else:
            resp = self._client.chat.completions.create(
                model=self.model, messages=messages, temperature=self.temperature)
        if resp.usage:
            self._in += resp.usage.prompt_tokens; self._out += resp.usage.completion_tokens
        if response_format is not None: return resp.choices[0].message.parsed
        return resp.choices[0].message.content

    def complete_vision(self, messages):
        """Send messages containing image_url content blocks to the vision model."""
        resp = self._client.chat.completions.create(
            model=self.model, messages=messages, temperature=self.temperature)
        if resp.usage:
            self._in += resp.usage.prompt_tokens; self._out += resp.usage.completion_tokens
        return resp.choices[0].message.content

    @property
    def total_cost(self):
        if self.model not in PRICES: return 0.0
        p = PRICES[self.model]
        return (self._in * p["input"] + self._out * p["output"]) / 1_000_000

llm = LLMClient(model="gpt-4o")

## Vision API Basics

A multimodal LLM processes images by splitting each image into fixed-size **patches** (typically $14 \times 14$ pixels each), encoding each patch through a vision encoder (often a ViT), and projecting the resulting patch embeddings into the same token space as text. These **patch tokens** are concatenated with the text token sequence before being fed to the language model, which can then attend across the full mixed sequence. From the API consumer's perspective, the image is simply another content block in the `messages` array.

The cost model for images follows from the patch count. GPT-4o charges approximately:

$$C_{\text{image}} = \begin{cases} 85 \text{ tokens} & \text{low detail} \\ 85 + 170 \cdot n_{\text{tiles}} \text{ tokens} & \text{high detail} \end{cases}$$

where $n_{\text{tiles}}$ is the number of $512 \times 512$ tiles the image is split into after rescaling to fit within a $2048 \times 2048$ bounding box. A typical earnings slide at $1280 \times 720$ requires $n_{\text{tiles}} = 4$, costing $85 + 680 = 765$ tokens. For comparison, a dense page of text is roughly 500 tokens. Vision adds significant cost at scale.

<br>

The message format for vision calls passes the image as a base64-encoded data URI inside a content block:

In [ ]:
# Vision message format (illustrative — not executed against the API)
vision_message_example = [
    {
        "role": "user",
        "content": [
            {
                "type": "image_url",
                "image_url": {
                    "url": "data:image/png;base64,<BASE64_STRING>",  # <1>
                    "detail": "high",                                 # <2>
                },
            },
            {
                "type": "text",
                "text": "What revenue figure is shown in this chart?",  # <3>
            },
        ],
    }
]

print(json.dumps(vision_message_example, indent=2)[:400], "...")

1. The `url` field accepts either a public HTTPS URL or an inline `data:image/...;base64,...` URI. For financial documents we always use base64 to avoid hosting images externally and to keep the pipeline self-contained.
2. `"detail": "high"` triggers the high-resolution tiling pass. Use `"low"` for thumbnails, icon recognition, or whenever spatial precision is not needed — it cuts image token cost by up to 20×.
3. Multiple content blocks are allowed in a single message. The model attends to image and text jointly.

We implement `encode_image` to convert any image file or in-memory bytes to a base64 data URI:

In [ ]:
def encode_image(source, mime: str = "image/png") -> str:
    """Convert a file path or bytes to a base64 data URI."""
    if isinstance(source, (str, os.PathLike)):
        with open(source, "rb") as f:
            data = f.read()
    else:
        data = source  # already bytes
    b64 = base64.b64encode(data).decode("utf-8")
    return f"data:{mime};base64,{b64}"


def vision_message(image_uri: str, question: str, detail: str = "high") -> list[dict]:
    """Build a single-turn vision message for LLMClient.complete_vision."""
    return [
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": image_uri, "detail": detail}},
                {"type": "text", "text": question},
            ],
        }
    ]


def estimate_image_tokens(width: int, height: int, detail: str = "high") -> int:
    """Estimate GPT-4o token cost for a single image."""
    if detail == "low":
        return 85
    # Rescale to fit within 2048x2048, then tile at 512x512
    scale = min(2048 / max(width, height), 1.0)
    w, h = int(width * scale), int(height * scale)
    tiles = ((w + 511) // 512) * ((h + 511) // 512)
    return 85 + 170 * tiles


for w, h, det in [(800, 600, "high"), (1280, 720, "high"), (400, 300, "low")]:
    t = estimate_image_tokens(w, h, det)
    cost_usd = t * PRICES["gpt-4o"]["input"] / 1_000_000
    print(f"{w}x{h} ({det:4s}): {t:4d} tokens  ≈ ${cost_usd:.5f}")

:::{.callout-note}
For a 100-page PDF processed at high detail, image tokens alone cost roughly $0.02–$0.05 per page with GPT-4o. At scale (thousands of filings), a **tiered strategy** — low-detail pass to classify page type, high-detail pass only on pages containing charts or tables — typically reduces costs by 60–70%.

:::

## Synthetic Financial Charts

We generate all chart images synthetically using matplotlib so that no external files are needed. The `FinancialChartGenerator` produces three chart types that commonly appear in earnings presentations: a grouped bar chart for quarterly revenue and EPS, a line chart for stock price with moving averages, and a pie chart for portfolio sector allocation. All charts are rendered to in-memory PNG bytes and returned as base64 data URIs ready for the vision API.

In [ ]:
#| code-fold: true
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np


class FinancialChartGenerator:
    """Generate synthetic financial charts as base64 PNG data URIs."""

    # Deterministic data for reproducibility
    QUARTERS = ["Q1 2024", "Q2 2024", "Q3 2024", "Q4 2024"]
    REVENUE  = [42.1, 45.8, 49.3, 53.7]  # billions USD
    EPS      = [3.21, 3.45, 3.89, 4.12]  # USD per share

    SECTORS = ["Technology", "Financials", "Healthcare", "Energy", "Consumer", "Other"]
    ALLOCS  = [28.4, 22.1, 15.7, 12.3, 13.5, 8.0]  # percent

    @staticmethod
    def _to_b64(fig) -> str:
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
        plt.close(fig)
        buf.seek(0)
        return encode_image(buf.read(), mime="image/png")

    @classmethod
    def earnings_bar_chart(cls) -> str:
        """Grouped bar chart: quarterly revenue (left axis) and EPS (right axis)."""
        fig, ax1 = plt.subplots(figsize=(8, 4))
        ax2 = ax1.twinx()

        x = np.arange(len(cls.QUARTERS))
        w = 0.35
        bars = ax1.bar(x - w/2, cls.REVENUE, w, color="#2196F3", label="Revenue ($B)")
        ax2.bar(x + w/2, cls.EPS, w, color="#FF9800", label="EPS ($)")

        ax1.set_xticks(x); ax1.set_xticklabels(cls.QUARTERS)
        ax1.set_ylabel("Revenue ($ billions)"); ax2.set_ylabel("EPS ($ per share)")
        ax1.set_ylim(0, 70); ax2.set_ylim(0, 6)
        ax1.yaxis.label.set_color("#2196F3"); ax2.yaxis.label.set_color("#FF9800")
        ax1.grid(linestyle="dotted", alpha=0.6)

        # Value labels
        for bar, rev in zip(bars, cls.REVENUE):
            ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f"${rev}B", ha="center", va="bottom", fontsize=8, color="#2196F3")

        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=8)
        fig.tight_layout()
        return cls._to_b64(fig)

    @classmethod
    def price_line_chart(cls) -> str:
        """Stock price line chart with 50-day and 200-day moving averages."""
        np.random.seed(42)
        n = 252  # one trading year
        returns = np.random.normal(0.0004, 0.015, n)
        price = 100 * np.cumprod(1 + returns)
        # Inject a golden cross at ~day 140
        price[140:] *= np.linspace(1.0, 1.15, n - 140)

        ma50  = np.convolve(price, np.ones(50)/50,  mode="valid")
        ma200 = np.convolve(price, np.ones(200)/200, mode="valid")
        days  = np.arange(n)

        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(days, price, color="#aaaaaa", lw=0.8, alpha=0.7, label="Price")
        ax.plot(days[49:],  ma50,  color="#2196F3", lw=2.0, label="50-day MA")
        ax.plot(days[199:], ma200, color="#F44336", lw=2.0, label="200-day MA")

        # Mark the golden cross
        cross_day = 199 + np.argmin(np.abs(ma50[150:] - ma200)) + 150 - 49
        ax.axvline(cross_day, color="#4CAF50", linestyle="dashed", lw=1.5, label=f"Golden cross (day {cross_day})")

        ax.set_xlabel("Trading Day"); ax.set_ylabel("Price ($)")
        ax.grid(linestyle="dotted", alpha=0.6)
        ax.legend(loc="upper left", fontsize=8)
        fig.tight_layout()
        return cls._to_b64(fig)

    @classmethod
    def sector_pie_chart(cls) -> str:
        """Sector allocation pie chart with percentage labels."""
        fig, ax = plt.subplots(figsize=(6, 5))
        colors = ["#2196F3", "#FF9800", "#4CAF50", "#F44336", "#9C27B0", "#607D8B"]
        wedges, texts, autotexts = ax.pie(
            cls.ALLOCS, labels=cls.SECTORS, colors=colors,
            autopct="%1.1f%%", startangle=140,
            wedgeprops={"edgecolor": "white", "linewidth": 1.5},
        )
        for at in autotexts:
            at.set_fontsize(8)
        ax.set_title("Portfolio Sector Allocation — FY2024", fontsize=11, fontweight="bold")
        fig.tight_layout()
        return cls._to_b64(fig)


gen = FinancialChartGenerator()
EARNINGS_CHART_URI = gen.earnings_bar_chart()
PRICE_CHART_URI    = gen.price_line_chart()
SECTOR_CHART_URI   = gen.sector_pie_chart()

print(f"earnings_chart : {len(EARNINGS_CHART_URI):,} chars")
print(f"price_chart    : {len(PRICE_CHART_URI):,} chars")
print(f"sector_chart   : {len(SECTOR_CHART_URI):,} chars")

Previewing the three charts:

In [ ]:
#| code-fold: true
from IPython.display import display, Image as IPImage

for uri, label in [
    (EARNINGS_CHART_URI, "Quarterly Earnings"),
    (PRICE_CHART_URI,    "Stock Price + Moving Averages"),
    (SECTOR_CHART_URI,   "Sector Allocation"),
]:
    raw = base64.b64decode(uri.split(",", 1)[1])
    print(f"--- {label} ---")
    display(IPImage(data=raw, format="png", width=600))

## Chart Q&A

We implement `ChartQA` to send a chart image together with a natural-language question to GPT-4o via `complete_vision`. For questions that require structured extraction — for example, pulling a specific numeric value — we also provide a Pydantic-backed `extract_value` method that returns a typed `ChartAnswer` with a `value`, `unit`, and `confidence` field.

In [ ]:
class ChartAnswer(BaseModel):
    value: str = Field(description="The extracted numeric or categorical value")
    unit: str  = Field(description="Unit of the value, e.g. 'billions USD', 'percent', 'USD/share'")
    confidence: Literal["high", "medium", "low"] = Field(
        description="Confidence in the extraction based on chart clarity"
    )
    reasoning: str = Field(description="One-sentence explanation of how the value was read")


class ChartQA:
    """Chart question-answering using GPT-4o vision."""

    SYSTEM = (
        "You are a financial analyst assistant. Answer questions about charts precisely. "
        "Read axis labels and numeric values carefully. Be concise."
    )

    def __init__(self, client: LLMClient):
        self._llm = client

    def ask(self, image_uri: str, question: str, detail: str = "high") -> str:  # <1>
        messages = [
            {"role": "system", "content": self.SYSTEM},
            *vision_message(image_uri, question, detail=detail),
        ]
        return self._llm.complete_vision(messages)

    def extract_value(self, image_uri: str, question: str) -> ChartAnswer:  # <2>
        prompt = (
            f"{question}\n\n"
            "Return a JSON object with fields: value, unit, confidence, reasoning."
        )
        messages = [
            {"role": "system", "content": self.SYSTEM},
            *vision_message(image_uri, prompt, detail="high"),
        ]
        raw = self._llm.complete_vision(messages)
        # Strip markdown code fences if present
        clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        return ChartAnswer.model_validate_json(clean)


qa = ChartQA(llm)

1. `ask` is the free-text interface: it prepends the analyst system prompt and forwards the vision message. The `detail` parameter is passed through to control cost — use `"low"` for quick classification questions, `"high"` for value extraction from dense charts.
2. `extract_value` prompts the model to return JSON and validates it against the `ChartAnswer` Pydantic model, giving callers a typed response rather than free text.

Running five questions against the earnings bar chart:

In [ ]:
EARNINGS_QA = [
    "What was the Q3 2024 revenue?",
    "Which quarter had the highest EPS, and what was the value?",
    "What was the quarter-over-quarter EPS growth from Q2 to Q3?",
    "What was the full-year 2024 revenue total across all four quarters?",
    "In which quarter did revenue first exceed $47 billion?",
]

print("=== Earnings Chart Q&A ===")
for q in EARNINGS_QA:
    ans = qa.ask(EARNINGS_CHART_URI, q)
    print(f"Q: {q}")
    print(f"A: {ans}\n")

Running five questions against the price and sector charts:

In [ ]:
PRICE_QA = [
    "When did the 50-day MA cross the 200-day MA (golden cross)?",
    "What was the approximate stock price at the golden cross?",
    "Was the overall price trend bullish or bearish over the year?",
]
SECTOR_QA = [
    "What is the largest sector allocation and what percentage does it represent?",
    "Which two sectors together account for more than 50% of the portfolio?",
]

print("=== Price Chart Q&A ===")
for q in PRICE_QA:
    ans = qa.ask(PRICE_CHART_URI, q)
    print(f"Q: {q}\nA: {ans}\n")

print("=== Sector Chart Q&A ===")
for q in SECTOR_QA:
    ans = qa.ask(SECTOR_CHART_URI, q)
    print(f"Q: {q}\nA: {ans}\n")

Demonstrating structured extraction with `ChartAnswer`:

In [ ]:
result = qa.extract_value(
    EARNINGS_CHART_URI,
    "What was Q3 2024 revenue in billions of dollars?"
)
print(f"value      : {result.value}")
print(f"unit       : {result.unit}")
print(f"confidence : {result.confidence}")
print(f"reasoning  : {result.reasoning}")

## Table Extraction from Scanned Documents

Scanned financial documents — regulatory filings, older annual reports, inter-office memos — arrive as image rasters rather than machine-readable text. Traditional OCR pipelines struggle with rotated tables, merged cells, and faint grid lines. GPT-4o vision reads these directly. We simulate a scanned financial table by rendering a pandas DataFrame as a matplotlib image, then adding Gaussian noise, slight rotation, and brightness variation to mimic scanning artifacts.

In [ ]:
#| code-fold: true
import pandas as pd
from matplotlib.patches import FancyBboxPatch
from scipy.ndimage import rotate as nd_rotate
import warnings; warnings.filterwarnings("ignore")


# Ground-truth financial table
FINANCIAL_TABLE = pd.DataFrame({
    "Metric":        ["Net Revenue", "Gross Profit", "Operating Income", "Net Income", "EPS (diluted)"],
    "FY2022 ($M)":   [38_420, 19_210, 8_640, 6_320, 2.84],
    "FY2023 ($M)":   [43_870, 22_150, 10_330, 7_890, 3.57],
    "FY2024 ($M)":   [50_310, 26_420, 13_180, 9_950, 4.52],
    "YoY Growth":    ["—", "—", "—", "—", "—"],  # computed below
})

# Compute YoY growth FY23→FY24
fy23 = [19_210, 22_150, 10_330, 7_890, 3.57]
fy24 = [26_420, 26_420, 13_180, 9_950, 4.52]
FINANCIAL_TABLE["YoY Growth"] = [
    f"+{(b/a - 1)*100:.1f}%" for a, b in zip(
        [38_420, 19_210, 8_640, 6_320, 2.84],
        [50_310, 26_420, 13_180, 9_950, 4.52],
    )
]


def render_table_as_scanned_image(df: pd.DataFrame, noise_level: float = 12.0) -> str:
    """Render a DataFrame as a matplotlib table image with scan-like noise."""
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.axis("off")

    tbl = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc="center",
        loc="center",
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1.2, 1.8)

    # Style header row
    for j in range(len(df.columns)):
        tbl[0, j].set_facecolor("#2196F3")
        tbl[0, j].set_text_props(color="white", fontweight="bold")

    ax.set_title("Apex Capital Group — Income Statement Summary",
                 fontsize=12, fontweight="bold", pad=12)
    fig.tight_layout()

    # Render to numpy array
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)

    # Add scanning artifacts
    from PIL import Image as PILImage
    img = PILImage.open(buf).convert("RGB")
    arr = np.array(img, dtype=np.float32)

    # Gaussian noise
    noise = np.random.normal(0, noise_level, arr.shape)
    arr = np.clip(arr + noise, 0, 255)

    # Slight brightness variation (scan unevenness)
    grad = np.linspace(0.95, 1.02, arr.shape[1])[np.newaxis, :, np.newaxis]
    arr = np.clip(arr * grad, 0, 255).astype(np.uint8)

    # Very slight rotation (0.4 degrees)
    arr = nd_rotate(arr, 0.4, reshape=False, cval=255)

    out_buf = io.BytesIO()
    PILImage.fromarray(arr.astype(np.uint8)).save(out_buf, format="PNG")
    return encode_image(out_buf.getvalue(), mime="image/png")


SCANNED_TABLE_URI = render_table_as_scanned_image(FINANCIAL_TABLE)
print(f"Scanned table image: {len(SCANNED_TABLE_URI):,} chars")

raw = base64.b64decode(SCANNED_TABLE_URI.split(",", 1)[1])
display(IPImage(data=raw, format="png", width=700))

We implement `TableExtractor` with two methods: `extract_to_dict` returns the raw JSON the model produces, and `extract_to_dataframe` wraps that into a pandas DataFrame:

In [ ]:
class TableExtractor:
    """Extract tabular data from a document image using GPT-4o vision."""

    SYSTEM = (
        "You are a financial data extraction assistant. "
        "Extract tables from document images exactly as they appear. "
        "Return only valid JSON — no markdown, no commentary."
    )

    PROMPT = (
        "Extract the table from this image as a JSON object with two keys:\n"
        '  "headers": list of column header strings\n'
        '  "rows": list of row lists (each row is a list of cell strings)\n'
        "Preserve all numeric values, units, and formatting exactly as shown."
    )

    def __init__(self, client: LLMClient):
        self._llm = client

    def extract_to_dict(self, image_uri: str) -> dict:  # <1>
        messages = [
            {"role": "system", "content": self.SYSTEM},
            *vision_message(image_uri, self.PROMPT, detail="high"),
        ]
        raw = self._llm.complete_vision(messages)
        clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        return json.loads(clean)

    def extract_to_dataframe(self, image_uri: str) -> pd.DataFrame:  # <2>
        data = self.extract_to_dict(image_uri)
        return pd.DataFrame(data["rows"], columns=data["headers"])


extractor = TableExtractor(llm)
extracted_df = extractor.extract_to_dataframe(SCANNED_TABLE_URI)
print("Extracted DataFrame:")
print(extracted_df.to_string(index=False))
print(f"\nShape: {extracted_df.shape}")

1. `extract_to_dict` sends the image and a schema-instructed prompt, then strips any markdown code fences the model may wrap the JSON in before parsing. We use `detail="high"` because table cell values require fine spatial resolution to read accurately.
2. `extract_to_dataframe` builds a DataFrame using the extracted headers and rows. If the model hallucinates a column header or misreads a cell value, the error will surface here as a shape mismatch, making it easy to add a validation step downstream.

We compare the extracted values to ground truth:

In [ ]:
#| code-fold: true
print("Ground truth vs. extracted (first three data columns):")
print(f"{'Metric':<20} {'GT FY2022':>12} {'Ex FY2022':>12}  {'GT FY2024':>12} {'Ex FY2024':>12}")
print("-" * 72)
for (_, gt_row), (_, ex_row) in zip(FINANCIAL_TABLE.iterrows(), extracted_df.iterrows()):
    gt_m = str(gt_row.iloc[0])[:18]
    gt22 = str(gt_row.iloc[1])
    ex22 = str(ex_row.iloc[1]) if len(ex_row) > 1 else "—"
    gt24 = str(gt_row.iloc[3])
    ex24 = str(ex_row.iloc[3]) if len(ex_row) > 3 else "—"
    print(f"{gt_m:<20} {gt22:>12} {ex22:>12}  {gt24:>12} {ex24:>12}")

:::{.callout-caution}
GPT-4o may reformat large numbers (e.g. `38_420` → `$38,420`) or interpret units differently from the original. Downstream pipelines that parse extracted values numerically must normalize separators and currency symbols before casting to `float`. Always validate extracted tables against known-good reference rows when available.

:::

## Document Understanding: PDF Pages as Images

The standard pattern for processing PDF financial reports is: (1) convert each page to a raster image using a library like `pdf2image` (which wraps `poppler`), (2) send each page to GPT-4o vision with a page-type-aware prompt, (3) aggregate the per-page extractions into a structured document model. We simulate a three-page earnings report — a cover page, an income statement table page, and a management outlook text page — using matplotlib-generated images, then process each with GPT-4o and assemble an `EarningsReport` Pydantic model.

In [ ]:
#| code-fold: true
def make_cover_page() -> str:
    """Generate a synthetic earnings presentation cover page."""
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.set_facecolor("#1A237E")
    fig.patch.set_facecolor("#1A237E")
    ax.axis("off")

    ax.text(0.5, 0.80, "APEX CAPITAL GROUP",
            transform=ax.transAxes, ha="center", va="center",
            fontsize=22, fontweight="bold", color="white")
    ax.text(0.5, 0.65, "Q4 & Full Year 2024 Earnings Presentation",
            transform=ax.transAxes, ha="center", va="center",
            fontsize=14, color="#90CAF9")
    ax.text(0.5, 0.50, "February 12, 2025",
            transform=ax.transAxes, ha="center", va="center",
            fontsize=11, color="#BBDEFB")
    ax.text(0.5, 0.35, "Full Year 2024 Revenue: $50.3B  |  Net Income: $9.95B  |  EPS: $4.52",
            transform=ax.transAxes, ha="center", va="center",
            fontsize=10, color="#E3F2FD",
            bbox=dict(boxstyle="round,pad=0.5", facecolor="#283593", edgecolor="#5C6BC0"))
    ax.text(0.5, 0.15,
            "CONFIDENTIAL — For investor use only. Not for redistribution.",
            transform=ax.transAxes, ha="center", va="center",
            fontsize=8, color="#7986CB", style="italic")

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight")
    plt.close(fig); buf.seek(0)
    return encode_image(buf.read(), mime="image/png")


def make_financials_page() -> str:
    """Generate a financials page: income statement table + quarterly earnings chart."""
    fig = plt.figure(figsize=(10, 6), constrained_layout=True)
    gs  = fig.add_gridspec(2, 1, height_ratios=[1.2, 1.8])

    ax_title = fig.add_subplot(gs[0])
    ax_title.axis("off")
    ax_title.text(0.5, 0.7, "Apex Capital Group — Financial Highlights FY2024",
                  transform=ax_title.transAxes, ha="center",
                  fontsize=13, fontweight="bold")

    tbl_data = [
        ["Net Revenue",      "$38.4B", "$43.9B", "$50.3B", "+14.7%"],
        ["Gross Profit",     "$19.2B", "$22.2B", "$26.4B", "+19.0%"],
        ["Operating Income", "$8.6B",  "$10.3B", "$13.2B", "+27.7%"],
        ["Net Income",       "$6.3B",  "$7.9B",  "$10.0B", "+26.1%"],
        ["EPS (diluted)",    "$2.84",  "$3.57",  "$4.52",  "+26.6%"],
    ]
    headers = ["Metric", "FY2022", "FY2023", "FY2024", "YoY Growth"]
    tbl = ax_title.table(cellText=tbl_data, colLabels=headers,
                         cellLoc="center", loc="lower center", bbox=[0, -1.6, 1, 1.3])
    tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.6)
    for j in range(len(headers)):
        tbl[0, j].set_facecolor("#1A237E")
        tbl[0, j].set_text_props(color="white", fontweight="bold")
    for i in range(1, len(tbl_data) + 1):
        bg = "#E8EAF6" if i % 2 == 0 else "white"
        for j in range(len(headers)):
            tbl[i, j].set_facecolor(bg)

    ax_chart = fig.add_subplot(gs[1])
    x = np.arange(4); w = 0.35
    ax2c = ax_chart.twinx()
    ax_chart.bar(x - w/2, FinancialChartGenerator.REVENUE, w, color="#2196F3", label="Revenue ($B)")
    ax2c.bar(x + w/2, FinancialChartGenerator.EPS, w, color="#FF9800", label="EPS ($)")
    ax_chart.set_xticks(x); ax_chart.set_xticklabels(FinancialChartGenerator.QUARTERS, fontsize=8)
    ax_chart.set_ylabel("Revenue ($B)", fontsize=8); ax2c.set_ylabel("EPS ($)", fontsize=8)
    ax_chart.set_ylim(0, 70); ax2c.set_ylim(0, 6)
    ax_chart.grid(linestyle="dotted", alpha=0.5)
    lines1, lab1 = ax_chart.get_legend_handles_labels()
    lines2, lab2 = ax2c.get_legend_handles_labels()
    ax_chart.legend(lines1 + lines2, lab1 + lab2, fontsize=7, loc="upper left")

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight")
    plt.close(fig); buf.seek(0)
    return encode_image(buf.read(), mime="image/png")


def make_outlook_page() -> str:
    """Generate a management commentary and outlook text page."""
    outlook_text = textwrap.dedent("""
        Management Commentary & FY2025 Outlook
        ────────────────────────────────────────────────────────────────────────────────

        FULL YEAR 2024 SUMMARY

        Apex Capital Group delivered record revenue of $50.3 billion for FY2024,
        representing 14.7% year-over-year growth. Net income grew 26.1% to $9.95 billion,
        driven by operating leverage as our expense ratio improved to 61.2% from 64.8%.
        EPS of $4.52 exceeded consensus estimates of $4.38 by 3.2%.

        Q4 2024 was particularly strong: revenue of $13.7 billion was up 18% YoY and
        up 7.3% sequentially. The Technology and Financials segments led performance,
        with combined revenue growth of 22% in the quarter.

        CAPITAL ALLOCATION

        The Board approved a $3.0 billion share repurchase program for FY2025, in
        addition to the regular quarterly dividend of $0.65 per share (annualized $2.60).
        Our CET1 ratio stands at 14.8%, well above our internal target of 13.0%.

        FY2025 GUIDANCE

        Management guides for FY2025 revenue in the range of $55–$58 billion (+9%–15%),
        EPS in the range of $5.00–$5.30, and operating margin expansion of 100–150 bps.
        We expect the Technology segment to grow 20%+ as AI-related product demand
        accelerates. Capital expenditure guidance is $2.8 billion.

        The outlook assumes no material deterioration in the macroeconomic environment
        and continued Federal Reserve policy stability.
    """).strip()

    fig, ax = plt.subplots(figsize=(9, 7))
    ax.axis("off")
    ax.text(0.05, 0.97, outlook_text, transform=ax.transAxes,
            ha="left", va="top", fontsize=8.5,
            fontfamily="monospace",
            wrap=True)
    fig.patch.set_facecolor("#FAFAFA")
    fig.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight")
    plt.close(fig); buf.seek(0)
    return encode_image(buf.read(), mime="image/png")


PAGE_URIS = [
    make_cover_page(),
    make_financials_page(),
    make_outlook_page(),
]
for i, uri in enumerate(PAGE_URIS, 1):
    print(f"Page {i}: {len(uri):,} chars")
    raw = base64.b64decode(uri.split(",", 1)[1])
    display(IPImage(data=raw, format="png", width=640))

We define the `EarningsReport` Pydantic model and the `EarningsPDFProcessor` that extracts structured data from each page and assembles the final report:

In [ ]:
class FinancialMetrics(BaseModel):
    fy2024_revenue_bn:     Optional[float] = Field(None, description="FY2024 net revenue in billions USD")
    fy2024_net_income_bn:  Optional[float] = Field(None, description="FY2024 net income in billions USD")
    fy2024_eps:            Optional[float] = Field(None, description="FY2024 diluted EPS in USD")
    yoy_revenue_growth_pct:Optional[float] = Field(None, description="FY2024 YoY revenue growth percent")
    cet1_ratio_pct:        Optional[float] = Field(None, description="CET1 capital ratio percent")


class FY2025Guidance(BaseModel):
    revenue_low_bn:    Optional[float] = Field(None, description="Lower bound of revenue guidance in billions")
    revenue_high_bn:   Optional[float] = Field(None, description="Upper bound of revenue guidance in billions")
    eps_low:           Optional[float] = Field(None, description="Lower bound of EPS guidance")
    eps_high:          Optional[float] = Field(None, description="Upper bound of EPS guidance")
    capex_bn:          Optional[float] = Field(None, description="Capital expenditure guidance in billions")
    commentary:        Optional[str]   = Field(None, description="Key qualitative guidance commentary")


class EarningsReport(BaseModel):
    company_name:    str
    report_period:   str
    presentation_date: Optional[str] = None
    metrics:         FinancialMetrics
    guidance:        FY2025Guidance
    key_highlights:  list[str] = Field(default_factory=list, description="3-5 bullet highlights")

Implementing `EarningsPDFProcessor`:

In [ ]:
class EarningsPDFProcessor:
    """Process a synthetic multi-page earnings PDF (as image URIs) into an EarningsReport."""

    PAGE_PROMPTS = [
        # Page 1: cover
        "Extract: company name, report period (e.g. 'Q4 & FY2024'), presentation date, "
        "and any headline financial figures shown. Return JSON.",
        # Page 2: financials
        "Extract all financial metrics from the table and chart: FY2024 revenue, net income, "
        "EPS, and YoY growth rates. Also extract quarterly revenue and EPS values. Return JSON.",
        # Page 3: outlook
        "Extract: (1) FY2025 revenue guidance range, (2) FY2025 EPS guidance range, "
        "(3) capex guidance, (4) CET1 ratio if mentioned, "
        "(5) three to five key management commentary highlights as bullet strings. Return JSON.",
    ]

    ASSEMBLY_PROMPT = (
        "You are a financial data assembler. Given per-page extractions from an earnings report, "
        "consolidate them into a single structured JSON matching this schema:\n"
        "  company_name, report_period, presentation_date, "
        "  metrics: {fy2024_revenue_bn, fy2024_net_income_bn, fy2024_eps, yoy_revenue_growth_pct, cet1_ratio_pct}, "
        "  guidance: {revenue_low_bn, revenue_high_bn, eps_low, eps_high, capex_bn, commentary}, "
        "  key_highlights: [list of strings]\n"
        "Use null for missing fields. Return only valid JSON."
    )

    def __init__(self, client: LLMClient):
        self._llm = client

    def process_pages(self, page_uris: list[str]) -> EarningsReport:  # <1>
        page_extractions = []
        for i, (uri, prompt) in enumerate(zip(page_uris, self.PAGE_PROMPTS), 1):
            print(f"  Processing page {i}/{len(page_uris)}...")
            msgs = [
                {"role": "system", "content": "You are a financial document extraction assistant. Return only JSON."},
                *vision_message(uri, prompt, detail="high"),
            ]
            raw = self._llm.complete_vision(msgs)
            page_extractions.append(raw)

        return self._assemble(page_extractions)  # <2>

    def _assemble(self, page_extractions: list[str]) -> EarningsReport:  # <3>
        combined = "\n\n".join(
            f"--- Page {i+1} extraction ---\n{text}"
            for i, text in enumerate(page_extractions)
        )
        messages = [
            {"role": "system", "content": self.ASSEMBLY_PROMPT},
            {"role": "user",   "content": combined},
        ]
        return self._llm.complete(messages, response_format=EarningsReport)


processor = EarningsPDFProcessor(llm)
print("Processing earnings report pages...")
report = processor.process_pages(PAGE_URIS)
print()
print(f"Company       : {report.company_name}")
print(f"Period        : {report.report_period}")
print(f"Date          : {report.presentation_date}")
print(f"FY2024 Revenue: ${report.metrics.fy2024_revenue_bn}B")
print(f"FY2024 EPS    : ${report.metrics.fy2024_eps}")
print(f"YoY Growth    : {report.metrics.yoy_revenue_growth_pct}%")
print(f"FY2025 Rev    : ${report.guidance.revenue_low_bn}B – ${report.guidance.revenue_high_bn}B")
print(f"FY2025 EPS    : ${report.guidance.eps_low} – ${report.guidance.eps_high}")
print("\nKey highlights:")
for h in report.key_highlights:
    print(f"  • {h}")

1. `process_pages` sends each page image with a page-type-specific extraction prompt. The prompts are ordered to match the page sequence in the synthetic deck: cover → financials → outlook.
2. After per-page extraction, the raw JSON strings are passed to `_assemble` for consolidation — this two-step approach avoids confusion between per-page extractions that may express the same metric in different formats.
3. `_assemble` invokes the text-only `complete` method with `response_format=EarningsReport` for structured parsing via `beta.chat.completions.parse`, which guarantees the response conforms to the Pydantic model.

## Multimodal RAG

Standard text RAG embeds text chunks with a sentence encoder and retrieves by cosine similarity. Extending it to images requires deciding how to represent image content in the embedding space. The practical approach is **caption-based embedding**: we generate a dense text caption of each image using GPT-4o vision, embed that caption with `text-embedding-3-small`, and store the embedding alongside the original image URI. At query time, the text query is embedded and compared to both text chunk embeddings and image-caption embeddings in a unified index. A `MultimodalChunk` carries a `type` field so the retriever knows whether to pass a text passage or an image URI to the LLM.

In [ ]:
class MultimodalChunk(BaseModel):
    id:           str
    type:         Literal["text", "image"]
    text:         str            # text content or GPT-4o caption for images
    image_uri:    Optional[str] = None   # base64 data URI (images only)
    metadata:     dict          = Field(default_factory=dict)


class MultimodalIndexer:
    """Embed and store both text and image chunks in a numpy index."""

    CAPTION_PROMPT = (
        "Describe this financial chart or document page in detail. "
        "Include all numeric values, axis labels, trends, and key observations. "
        "Be comprehensive — this caption will be used for semantic search."
    )

    def __init__(self, client: LLMClient):
        self._llm    = client
        self._oai    = openai.OpenAI()
        self._chunks : list[MultimodalChunk] = []
        self._embeds : list[np.ndarray]      = []

    def _embed(self, text: str) -> np.ndarray:  # <1>
        resp = self._oai.embeddings.create(model="text-embedding-3-small", input=[text])
        return np.array(resp.data[0].embedding, dtype=np.float32)

    def _caption(self, image_uri: str) -> str:  # <2>
        msgs = [
            {"role": "system", "content": "You are a financial analyst. Be precise and exhaustive."},
            *vision_message(image_uri, self.CAPTION_PROMPT, detail="high"),
        ]
        return self._llm.complete_vision(msgs)

    def add_text(self, text: str, chunk_id: str, metadata: dict | None = None) -> None:
        chunk = MultimodalChunk(id=chunk_id, type="text", text=text, metadata=metadata or {})
        self._chunks.append(chunk)
        self._embeds.append(self._embed(text))

    def add_image(self, image_uri: str, chunk_id: str, metadata: dict | None = None) -> None:
        caption = self._caption(image_uri)
        chunk = MultimodalChunk(id=chunk_id, type="image", text=caption,
                                image_uri=image_uri, metadata=metadata or {})
        self._chunks.append(chunk)
        self._embeds.append(self._embed(caption))

    def get_index(self) -> tuple[list[MultimodalChunk], np.ndarray]:
        return self._chunks, np.stack(self._embeds)

1. We embed with `text-embedding-3-small` (1536 dimensions, $0.02/1M tokens) for both text and image captions, keeping all chunk representations in the same vector space. A production system could use a dedicated multimodal encoder like CLIP, but caption-based embedding is more practical with the OpenAI API and produces strong retrieval quality for financial content.
2. `_caption` runs a high-detail vision call with an exhaustive description prompt. The quality of retrieval for image chunks is entirely determined by caption quality — verbose, fact-dense captions retrieve better than terse summaries.

We implement `MultimodalRetriever` to query the unified index:

In [ ]:
class MultimodalRetriever:
    """Cosine-similarity retrieval over a multimodal index."""

    def __init__(self, chunks: list[MultimodalChunk], embeddings: np.ndarray):
        self._chunks = chunks
        # L2-normalise for cosine via dot product
        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        self._embeds = embeddings / (norms + 1e-9)
        self._oai = openai.OpenAI()

    def _embed_query(self, query: str) -> np.ndarray:
        resp = self._oai.embeddings.create(model="text-embedding-3-small", input=[query])
        v = np.array(resp.data[0].embedding, dtype=np.float32)
        return v / (np.linalg.norm(v) + 1e-9)

    def retrieve(
        self,
        query: str,
        k: int = 4,
        type_filter: Optional[Literal["text", "image"]] = None,  # <1>
    ) -> list[tuple[MultimodalChunk, float]]:
        qvec = self._embed_query(query)
        scores = self._embeds @ qvec

        if type_filter is not None:  # <2>
            mask = np.array([c.type == type_filter for c in self._chunks])
            scores = np.where(mask, scores, -np.inf)

        top_idx = np.argsort(scores)[::-1][:k]
        return [(self._chunks[i], float(scores[i])) for i in top_idx if scores[i] > -np.inf]

1. `type_filter` lets callers restrict retrieval to text-only or image-only chunks. This is useful when the query explicitly involves a chart (e.g. "show me the revenue chart") versus a factual text lookup.
2. Masked-out chunks receive $-\infty$ so they sort last and are excluded from the top-$k$ slice without needing a separate index.

We index the three earnings report pages (as image chunks) together with four text chunks drawn from the management commentary:

In [ ]:
TEXT_CHUNKS = [
    {
        "id": "txt-summary",
        "text": (
            "Apex Capital Group delivered record revenue of $50.3 billion for FY2024, "
            "representing 14.7% year-over-year growth. Net income grew 26.1% to $9.95 billion, "
            "driven by operating leverage as the expense ratio improved to 61.2% from 64.8%."
        ),
        "metadata": {"source": "outlook_page", "section": "summary"},
    },
    {
        "id": "txt-capital",
        "text": (
            "The Board approved a $3.0 billion share repurchase program for FY2025, in addition "
            "to the regular quarterly dividend of $0.65 per share (annualized $2.60). "
            "Our CET1 ratio stands at 14.8%, well above our internal target of 13.0%."
        ),
        "metadata": {"source": "outlook_page", "section": "capital"},
    },
    {
        "id": "txt-guidance",
        "text": (
            "Management guides for FY2025 revenue in the range of $55–$58 billion (+9%–15%), "
            "EPS in the range of $5.00–$5.30, and operating margin expansion of 100–150 bps. "
            "Capital expenditure guidance is $2.8 billion."
        ),
        "metadata": {"source": "outlook_page", "section": "guidance"},
    },
    {
        "id": "txt-tech",
        "text": (
            "We expect the Technology segment to grow 20%+ as AI-related product demand accelerates. "
            "Q4 2024 revenue of $13.7 billion was up 18% YoY and 7.3% sequentially. "
            "The Technology and Financials segments combined grew 22% in Q4."
        ),
        "metadata": {"source": "outlook_page", "section": "segments"},
    },
]

indexer = MultimodalIndexer(llm)

print("Indexing text chunks...")
for chunk in TEXT_CHUNKS:
    indexer.add_text(chunk["text"], chunk["id"], chunk["metadata"])
    print(f"  indexed {chunk['id']}")

print("\nIndexing image chunks (captioning via GPT-4o)...")
page_labels = ["img-cover", "img-financials", "img-outlook"]
for page_id, uri in zip(page_labels, PAGE_URIS):
    indexer.add_image(uri, page_id, metadata={"source": "earnings_deck", "page": page_id})
    print(f"  indexed {page_id}")

mm_chunks, mm_embeds = indexer.get_index()
retriever = MultimodalRetriever(mm_chunks, mm_embeds)
print(f"\nIndex size: {len(mm_chunks)} chunks ({mm_embeds.shape[1]}d embeddings)")

We run two queries to confirm cross-modal retrieval:

In [ ]:
queries = [
    ("What guidance did management give for FY2025 EPS?", None),
    ("quarterly revenue and EPS bar chart breakdown",     "image"),
]

for query, tfilter in queries:
    label = f"(type_filter='{tfilter}')" if tfilter else "(no filter)"
    print(f"\nQuery: {query!r} {label}")
    results = retriever.retrieve(query, k=3, type_filter=tfilter)
    for chunk, score in results:
        snippet = chunk.text[:100].replace("\n", " ")
        print(f"  [{score:.3f}] [{chunk.type:5s}] {chunk.id:20s}  {snippet}...")

## Cost and Latency Analysis

Processing financial documents with vision imposes meaningfully higher cost and latency than text-only pipelines. The tradeoff is not always favorable: many PDF parsers (`pdfplumber`, `pymupdf`, `camelot`) extract text and tables from digital PDFs with near-perfect fidelity at essentially zero cost. Vision is justified when (1) the PDF is a scanned raster, (2) the document contains charts whose data is not otherwise machine-readable, or (3) layout and spatial context matter to interpretation.

The table below summarizes the cost and latency profile per page for three processing strategies:

In [ ]:
#| code-fold: true
STRATEGIES = [
    {
        "Strategy":       "Text parser only",
        "Image tokens":   0,
        "Text tokens":    "~500",
        "Input cost/page": "$0.00025",
        "p50 latency":    "<50 ms",
        "p99 latency":    "<200 ms",
        "Use case":       "Digital PDFs with clean text/tables",
    },
    {
        "Strategy":       "Vision — low detail",
        "Image tokens":   85,
        "Text tokens":    "~500",
        "Input cost/page": "$0.00147",
        "p50 latency":    "1.5 s",
        "p99 latency":    "4 s",
        "Use case":       "Page classification, rough content triage",
    },
    {
        "Strategy":       "Vision — high detail (1280×720)",
        "Image tokens":   765,
        "Text tokens":    "~500",
        "Input cost/page": "$0.00316",
        "p50 latency":    "3 s",
        "p99 latency":    "8 s",
        "Use case":       "Charts, scanned tables, layout understanding",
    },
    {
        "Strategy":       "Vision — high detail (2048×2048)",
        "Image tokens":   1445,
        "Text tokens":    "~500",
        "Input cost/page": "$0.00486",
        "p50 latency":    "5 s",
        "p99 latency":    "12 s",
        "Use case":       "Dense charts, fine print, regulatory exhibits",
    },
]

df_costs = pd.DataFrame(STRATEGIES)
print(df_costs.to_string(index=False))

We implement a lightweight `PageClassifier` that uses a low-detail vision pass to decide whether a page needs a full high-detail extraction, reducing per-page cost for text-heavy pages:

In [ ]:
class PageType(BaseModel):
    page_type: Literal["cover", "text", "table", "chart", "mixed"]
    needs_high_detail: bool
    reason: str


class PageClassifier:
    """Classify page type with a low-detail vision pass to guide downstream strategy."""

    PROMPT = (
        "Classify this document page. Respond with JSON: "
        '{"page_type": one of [cover|text|table|chart|mixed], '
        '"needs_high_detail": true/false, '
        '"reason": one sentence}. '
        "Set needs_high_detail=true only if the page contains charts, tables, or dense numeric data."
    )

    def __init__(self, client: LLMClient):
        self._llm = client

    def classify(self, image_uri: str) -> PageType:
        msgs = [
            {"role": "system", "content": "You are a document layout classifier. Return only JSON."},
            *vision_message(image_uri, self.PROMPT, detail="low"),  # <1>
        ]
        raw = self._llm.complete_vision(msgs)
        clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        return PageType.model_validate_json(clean)


classifier = PageClassifier(llm)
page_names = ["Cover page", "Financials page", "Outlook page"]

print("Page classification (low-detail pass):")
for name, uri in zip(page_names, PAGE_URIS):
    pt = classifier.classify(uri)
    flag = "→ HIGH" if pt.needs_high_detail else "→ LOW "
    print(f"  {name:<18} [{pt.page_type:7s}] {flag}  {pt.reason}")

1. The classification call uses `detail="low"` (85 tokens) rather than `detail="high"` (765 tokens). Since we only need to identify whether charts or tables are present — not read their values — low detail is sufficient and reduces the triage cost by 9×.

:::{.callout-tip}
In production financial document pipelines, run a PDF text-extraction pass first with `pdfplumber` or `pymupdf`. Only fall back to GPT-4o vision for pages where text extraction yields fewer than 50 characters — this covers scanned pages, image-embedded charts, and encrypted PDFs while keeping the average per-page cost close to the text-parser tier.

:::

## End-to-End Demo

We now put all components together in `EarningsAnalystAssistant`. It holds a reference to the multimodal retriever built above, generates context-aware messages that include both retrieved text passages and retrieved image chunks, and produces grounded answers. The assistant handles two classes of questions: (1) quantitative questions that require reading charts or tables, and (2) qualitative questions about management commentary.

In [ ]:
class EarningsAnalystAssistant:
    """Answer analyst questions using multimodal RAG over an earnings presentation."""

    SYSTEM = (
        "You are a senior equity research analyst. Answer questions about earnings reports "
        "precisely and concisely. Cite the source of each figure (text passage or chart). "
        "If a value is not clearly visible or stated, say so."
    )

    def __init__(self, retriever: MultimodalRetriever, client: LLMClient):
        self._retriever = retriever
        self._llm = client

    def answer(self, question: str, k: int = 3) -> str:
        results = self._retriever.retrieve(question, k=k)  # <1>

        # Build a mixed content array
        content: list[dict] = []
        text_context_parts = []

        for i, (chunk, score) in enumerate(results):
            if chunk.type == "text":  # <2>
                text_context_parts.append(f"[Text {i+1} | {chunk.id}]\n{chunk.text}")
            else:  # image chunk
                text_context_parts.append(f"[Image {i+1} | {chunk.id}] See attached image below.")

        context_preamble = "\n\n".join(text_context_parts)
        content.append({"type": "text", "text": f"Retrieved context:\n\n{context_preamble}\n\nQuestion: {question}"})

        for chunk, _ in results:  # <3>
            if chunk.type == "image" and chunk.image_uri:
                content.append({"type": "image_url", "image_url": {"url": chunk.image_uri, "detail": "high"}})

        messages = [
            {"role": "system", "content": self.SYSTEM},
            {"role": "user",   "content": content},
        ]
        return self._llm.complete_vision(messages)


assistant = EarningsAnalystAssistant(retriever, llm)

1. We retrieve without a type filter so both text and image chunks compete on relevance. Questions about charts naturally retrieve image chunks (because the query matches their verbose captions); questions about guidance retrieve text chunks.
2. Text chunks contribute only to the preamble string, which the model reads as ordinary context. Image chunks contribute both a placeholder in the preamble (`"See attached image below"`) and an `image_url` content block in the actual message, so the model can attend to the chart directly.
3. Images are appended after the text preamble content block. The OpenAI API accepts multiple content blocks of mixed types in a single user message.

Running five analyst questions — three that require charts and two that require text:

In [ ]:
ANALYST_QUESTIONS = [
    # Require chart reading
    "What was the year-over-year revenue growth from FY2022 to FY2024?",
    "What was Q3 2024 revenue and EPS, and how did they compare to Q2 2024?",
    "Which quarter in 2024 showed the strongest EPS acceleration?",
    # Require text reading
    "What guidance did management give for next year's revenue and EPS?",
    "What is the company's CET1 ratio and how does it compare to its internal target?",
]

for i, question in enumerate(ANALYST_QUESTIONS, 1):
    print(f"Q{i}: {question}")
    answer = assistant.answer(question)
    print(f"A:  {answer}")
    print()

Summarizing token usage and cost for the full notebook session:

In [ ]:
print(f"Session token usage ({llm.model}):")
print(f"  Prompt tokens     : {llm._in:,}")
print(f"  Completion tokens : {llm._out:,}")
print(f"  Total tokens      : {llm._in + llm._out:,}")
print(f"  Estimated cost    : ${llm.total_cost:.4f}")

:::{.callout-important}
The `complete_vision` method bypasses the `response_format` structured-output path. OpenAI's `beta.chat.completions.parse` endpoint does not currently support `image_url` content blocks. For structured extraction from images, use the standard `create` endpoint (as `complete_vision` does) and parse the JSON response manually — as demonstrated in `extract_value` and `TableExtractor.extract_to_dict`.

:::

## Appendix: Decision Framework for Financial Document Processing

The choice between text extraction and vision is driven by document type, content class, and throughput requirements. The table below summarizes the recommended strategy for common financial document types encountered in production:

| Document type | Content class | Recommended strategy |
|:--|:--|:--|
| Digital SEC 10-K / 10-Q | Text + structured tables | PDF parser (e.g. `pdfplumber`) |
| Scanned legacy filing | Text + tables (rasterized) | GPT-4o vision, high detail |
| Earnings presentation deck | Charts + text + tables | Page classification → tiered detail |
| Bloomberg/FactSet export | Clean structured data | Direct parsing, no LLM needed |
| Analyst research PDF | Mixed text + charts | PDF parser + vision for chart pages |
| Handwritten notes / annotations | Freeform text | GPT-4o vision, high detail |
| Infographic / data visualization | Charts only | GPT-4o vision + structured extraction |

: Recommended processing strategy by document type. {tbl-colwidths="[30,30,40]"}

<br>

The decision logic for any individual page is:

1. **Try text extraction first.** If `pdfplumber` returns more than 100 characters for a page, treat it as a digital page and use text-only RAG.
2. **Classify with low-detail vision.** For pages with sparse text extraction, run `PageClassifier` at `detail="low"` (85 tokens) to determine page type.
3. **Apply high-detail vision selectively.** Only chart, table, and mixed pages receive a full `detail="high"` pass. Cover and pure-text pages that failed text extraction can use `detail="low"` or a cheaper OCR fallback.
4. **Cache captions.** Image captions generated for multimodal RAG indexing are expensive to produce. Persist them alongside the image hash so repeated indexing runs skip re-captioning.

The cost ceiling for the highest-throughput scenario — a 200-page annual report processed entirely at high-detail vision — is approximately $200 × $0.003 = **$0.60 per filing** with GPT-4o. Applying the tiered strategy to a typical annual report (80% text pages, 20% chart/table pages) reduces this to roughly **$0.15 per filing**.

---

$\blacksquare$